# **Lab: Embeddings and Data Preparation for LLMs**

This notebook implements the data preparation pipeline for a Large Language Model, following Chapter 2 of *Build a Large Language Model (From Scratch)*.

It covers:
1. Loading and Tokenizing Text
2. Creating Input-Target Pairs (Dataloaders)
3. Implementing Token and Positional Embeddings

## **Set Up**

In [2]:
%pip install torch tiktoken numpy matplotlib

  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl.metadata (60 kB)
  Using cached matplotlib-3.9.4-cp39-cp39-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached contourpy-1.3.0-cp39-cp39-macosx_11_0_arm64.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.2-cp39-cp39-macosx_10_9_universal2.whl.metadata (113 kB)
  Using cached kiwisolver-1.4.7-cp39-cp39-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pillow-11.3.0-cp39-cp39-macosx_11_0_arm64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 39.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.1/997.1 kB 30.7 MB/s  0:00:00
Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
Using cached matplotlib-3.9.4-cp39-cp39-macosx_11_0_arm64.whl (7.8 MB)
Usin

In [3]:
import re
import torch
import tiktoken
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.8.0
tiktoken version: 0.12.0


### **File Load**

In [8]:
import os

if not os.path.exists("the-verdict.txt"):
    raise FileNotFoundError("Please download 'the-verdict.txt' and place it in the same directory.")

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters:", len(raw_text))
print(f"The first 200 characters:\n{raw_text[:200]}")

Total characters: 20479
The first 200 characters:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


## **Part 1. Tokenization**

### **Why this step matters for LLMs & Agentic Systems?**

Tokenization is the non-negotiable first step of any NLP pipeline. It is the process of converting raw text strings into a sequence of integers that a Neural Network can process.

**For Agentic Systems:**

The choice of tokenizer dictates the model's ability to reason about structure and code.
1.  **Code Generation & JSON:** Agents often output JSON or Python code. Modern tokenizers (BPE) are optimized to treat common code constructs (like `    def`, `return`, or `":`) as single tokens. If a tokenizer splits these inconsistently, the agent struggles to follow syntax rules.
2.  **Arithmetic Reasoning:** Poor tokenization is why early LLMs failed at math. If "1024" is tokenized as `[10, 24]` in one context and `[1, 02, 4]` in another, the model cannot learn consistent arithmetic rules.

In this lab we compare to approaches:

### **Simple Tokenization**
- **Mechanism:** 
  We simply split text by spaces and punctuation. `["The", "cat", "sat"]`.
- **The Problem:** 
  This approach scales poorly. To cover the English language, you would need a vocabulary of millions of tokens. Furthermore, "cat" and "cats" are treated as completely unrelated IDs. The model has to relearn the concept of "plurality" for every single noun.

In [22]:
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(f"Total tokens: {len(preprocessed)}")
print(f"First 50 tokens: {preprocessed[:50]}")

all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(f"Vocabulary size: {vocab_size}")

vocab = {token: integer for integer, token in enumerate(all_words)}

print(f"First 30 vocabulary entries: {list(vocab.items())[:30]}")

Total tokens: 4649
First 50 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself']
Vocabulary size: 1159
First 30 vocabulary entries: [('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4), (',', 5), ('--', 6), ('.', 7), (':', 8), (';', 9), ('?', 10), ('A', 11), ('Ah', 12), ('Among', 13), ('And', 14), ('Are', 15), ('Arrt', 16), ('As', 17), ('At', 18), ('Be', 19), ('Begin', 20), ('Burlington', 21), ('But', 22), ('By', 23), ('Carlo', 24), ('Carlo;', 25), ('Chicago', 26), ('Claude', 27), ('Come', 28), ('Croft', 29)]


In [23]:
class TextTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer = TextTokenizer(vocab)
text = "I glanced after him, struck by his last word. Victor Grindle was, in fact, becoming the man of the moment--as Jack himself, one might put it, had been the man of the hour"
ids = tokenizer.encode(text)
print(f"Encoded: {ids}")
print(f"Decoded: {tokenizer.decode(ids)}")

Encoded: [55, 501, 143, 555, 5, 968, 248, 559, 615, 1145, 7, 109, 43, 1104, 5, 579, 428, 5, 212, 1013, 671, 738, 1013, 696, 6, 182, 59, 557, 5, 745, 686, 826, 596, 5, 522, 214, 1013, 671, 738, 1013, 568]
Decoded: I glanced after him, struck by his last word. Victor Grindle was, in fact, becoming the man of the moment -- as Jack himself, one might put it, had been the man of the hour


### **Using tiktoken**

- **Mechanism:** 
  Used by GPT-2, GPT-4, and Llama. Instead of splitting by word, it iteratively merges the most frequent adjacent characters.
- **The Solution:**
  - Fixed Vocabulary:
    BPE allows us to cap the vocabulary size while still being able to represent any string.
  - Subword Semantics:
    It breaks complex words into meaningful chunks. "Walking" becomes `["walk", "ing"]`. The model learns that `ing` implies "present continuous tense" and applies that knowledge to any verb it sees.
  - No "Unknown" Words:
    If a word is rare, BPE degrades gracefully into individual characters. It never crashes on unknown input.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(f"Encoded text: {integers}")
print(f"Decoded text: {tokenizer.decode(integers)}")
print(f"GPT-2 vocabulary size: {tokenizer.n_vocab}")

enc_text = tokenizer.encode(raw_text)
print(f"\nTotal tokens in text: {len(enc_text)}")
print(f"First 50 token IDs: {enc_text[:30]}")

Encoded text: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Decoded text: Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.
GPT-2 vocabulary size: 50257

Total tokens in text: 5145
First 50 token IDs: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284, 3285]


### **How is this related to Neural Network concepts?**

Tokenization is effectively a Compression and Dimensionality Reduction step.

- **Input Layer Efficiency:** The size of your Neural Network's embedding layer is directly determined by the Vocabulary Size. A naive word-level tokenizer would require a massive $V$ (millions of rows), making the model impossibly large and slow to train.
- **Information Density:** BPE optimizes the "information per token." By compressing frequent phrases into single integers, we allow the fixed-size context window of the Transformer (e.g., 1024 tokens) to hold more actual information (more sentences/paragraphs).

___
## **Part 2. Data Sampling with Sliding Window**

### **Why this step matters for LLMs & Agentic Systems**

Now that we have tokens, we need to structure them into training examples. LLMs are trained on a simple objective: Given a sequence of words, predict the very next word.

**For Agentic Systems:**

The quality of this "sliding window" training directly impacts an agent's ability to maintain Coherence and Context.
1.  **Context Window:** An agent has a limited memory, so,  the `max_length` parameter we define below simulates this limit. If we train with a short window, the model will never learn to reference information from 3 paragraphs ago.
2.  **Instruction Following:** By seeing thousands of overlapping examples, the model learns the structure of commands. It learns that "calculate" is usually followed by numbers, or "summary" is followed by a condensed text.

### **The Sliding Window Mechanism**

We don't just feed the book into the model linearly. We use a Sliding Window to create multiple examples from a single sentence.

**The Concept:**

Let's take as an example following the sentence: "The quick brown fox"
- **Input 1:** `[The, quick, brown]` $\rightarrow$ **Target:** `fox`
- **Input 2:** `[quick, brown, fox]` $\rightarrow$ **Target:** `jumps` (if the sentence continued)

**Key Parameters:**
- **`max_length`:** How many tokens the model can see at once. It´s basically the size of the context
- **`stride`:** How many steps we move the window for the next example.
  -  Small Stride (High Overlap): Creates more training data. We see the same sentence shifted slightly, forcing the model to understand the same words in slightly different positions.
  - Large Stride (No Overlap): Faster to process, but we lose the rich "contextual variation" that helps the model generalize.

In [25]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [26]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True,num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

### **Testing Data Loader**

In [28]:
dataloader = create_dataloader_v1(
    raw_text, 
    batch_size=8, 
    max_length=4, 
    stride=2, 
    shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)

print("First batch input:", first_batch[0])
print("First batch target:", first_batch[1])

tokenizer = tiktoken.get_encoding("gpt2")
input_text = tokenizer.decode(first_batch[0][0].tolist())
target_text = tokenizer.decode(first_batch[1][0].tolist())

print(f"\nInput text: '{input_text}'")
print(f"Target text: '{target_text}'")

First batch input: tensor([[   40,   367,  2885,  1464],
        [ 2885,  1464,  1807,  3619],
        [ 1807,  3619,   402,   271],
        [  402,   271, 10899,  2138],
        [10899,  2138,   257,  7026],
        [  257,  7026, 15632,   438],
        [15632,   438,  2016,   257],
        [ 2016,   257,   922,  5891]])
First batch target: tensor([[  367,  2885,  1464,  1807],
        [ 1464,  1807,  3619,   402],
        [ 3619,   402,   271, 10899],
        [  271, 10899,  2138,   257],
        [ 2138,   257,  7026, 15632],
        [ 7026, 15632,   438,  2016],
        [  438,  2016,   257,   922],
        [  257,   922,  5891,  1576]])

Input text: 'I HAD always'
Target text: ' HAD always thought'


### **How is this related to Neural Network concepts?**

This step transforms raw data into Supervised Learning pairs (X, y).
- **Input (X):** A tensor of shape `(Batch_Size, Max_Length)`.
- **Target (y):** A tensor of the same shape, but shifted by one position.
- **Loss Calculation:** The Neural Network calculates the Cross-Entropy Loss between its prediction for position t and the actual token at t+1. This scalar loss is what drives the Gradient Descent.

___
## **Part 3. Creating Token and Positional Embeddings**

### **Why this step matters for LLMs & Agentic Systems**

This code block performs the final transformation of our data before it enters the Transformer. We are converting our integer sequences (which the computer sees as arbitrary numbers) into rich, dense vectors that encode both Meaning** and Order.

**For Agentic Systems:**

It matters for Contextual Awareness because, by summing the token and positional vectors, we create a composite representation. This allows an agent to understand that the word "delete" at the *start* of a sentence (a command) has a different vector representation than "delete" at the *end* (a description). This distinction is vital for agents to follow instructions accurately without hallucinating intent.

In [ ]:
import torch.nn as nn

vocab_size = 50257      # GPT-2 vocabulary size
output_dim = 256        # Embedding dimensionality (768 for real GPT-2, 256 for this lab)
context_length = 1024   # Max context length (block_size)

torch.manual_seed(123)
token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = nn.Embedding(context_length, output_dim)

print(f"Token Embedding Weight Matrix: {token_embedding_layer.weight.shape}")
print(f"Positional Embedding Weight Matrix: {pos_embedding_layer.weight.shape}")

Token Embedding Weight Matrix: torch.Size([50257, 256])
Positional Embedding Weight Matrix: torch.Size([1024, 256])


### **Dataloader**

In [31]:
batch = next(iter(dataloader))
input_ids = batch[0]

print(f"Input Batch Shape: {input_ids.shape}")

token_embeddings = token_embedding_layer(input_ids)
print(f"Token Embeddings Shape: {token_embeddings.shape}")

pos_ids = torch.arange(input_ids.shape[1]) 
pos_embeddings = pos_embedding_layer(pos_ids)
print(f"Positional Embeddings Shape: {pos_embeddings.shape}")

input_embeddings = token_embeddings + pos_embeddings

print(f"Final Input Embeddings Shape: {input_embeddings.shape}")

Input Batch Shape: torch.Size([8, 4])
Token Embeddings Shape: torch.Size([8, 4, 256])
Positional Embeddings Shape: torch.Size([4, 256])
Final Input Embeddings Shape: torch.Size([8, 4, 256])


### **What is happening in the code?**

We are performing three distinct operations in this block:

**1. Initialization (`nn.Embedding`)**
- We define two lookup tables.
  - `token_embedding_layer`: A matrix of size 50,257 x 256. Each row represents the specific semantic meaning of a word in our vocabulary.
  - `pos_embedding_layer`: A matrix of size 1,024 x 256. Each row represents the geometric "signature" of a specific position in the sequence.

**2. The Lookup (Forward Pass)**
- `token_embeddings = token_embedding_layer(input_ids)`: The model replaces every integer in the batch with its corresponding row from the weight matrix.
- NN Concept: This is computationally equivalent to multiplying a One-Hot encoded vector by a Dense Weight Matrix, but optimized for speed.

**3. The Combination (`+`)**
- `input_embeddings = token_embeddings + pos_embeddings`: We add the two vectors together.
- Broadcasting: Notice that `pos_embeddings` (Shape: `Length x Dim`) is smaller than `token_embeddings` (Shape: `Batch x Length x Dim`). PyTorch automatically "broadcasts" the position vectors, adding the *same* position vector to every sample in the batch.
- Result: The final tensor contains information about *what* the word is (from the token embedding) and *where* it is (from the positional embedding), ready for the Self-Attention mechanism.

### **How is this related to Neural Network Concepts?**

**1. Lookup = Matrix Multiplication**
The `token_embedding_layer` is effectively a Linear Layer without bias. Mathematically, looking up an index is identical to multiplying a "One-Hot" vector by the weight matrix, just optimized for speed.

**2. Trainable Weights (Backpropagation)**
The embedding matrices are parameters of the network initialized with random numbers. During training, Backpropagation updates these specific numbers to minimize prediction error, effectively "learning" the meaning of words.

**3. Broadcasting & Addition**
In `input_embeddings = token_embeddings + pos_embeddings`, we use Broadcasting to add the single position matrix to every sample in the batch. In high-dimensional space, adding vectors superimposes the "position" info onto the "word" info without destroying it.